# Exp 2 — fruit still life: Gemini app vs Interactions API

Re-runs the exp 2 edit chain with Nano Banana 2 (`gemini-3.1-flash-image`) through the
multi-turn Interactions API — 2K output, high thinking level. The chain starts from
`data/exp2/0.jpeg` (the image generated in the Gemini app) and applies the same
10 edits, each turn building on the previous interaction. Outputs are saved to
`data/exp2_api/`.

In [1]:
import base64
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv
from google import genai

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GOOGLE_GEMINI_API_KEY"])

MODEL = "gemini-3.1-flash-image"  # Nano Banana 2
RESPONSE_FORMAT = {"type": "image", "mime_type": "image/jpeg", "aspect_ratio": "16:9", "image_size": "2K"}
GENERATION_CONFIG = {"thinking_level": "high"}

APP_DIR = Path("../data/exp2")      # Gemini app outputs: 0.jpeg (start) ... 10.jpeg
API_DIR = Path("../data/exp2_api")  # Interactions API outputs, same numbering
API_DIR.mkdir(exist_ok=True)

In [2]:
EDITS = [
    'Add a single ripe yellow banana on the table to the left of the red apple. Keep the apple unchanged and preserve the white background, table, lighting, camera angle, shadows, and overall composition exactly as they are.',
    'Add a single realistic orange on the table to the right of the red apple. Preserve the apple, the banana, and everything else exactly as they are, including the table, white background, lighting, shadows, and camera angle.',
    'Add a single green pear behind the red apple, slightly offset so it remains visible. Preserve the apple, banana, orange, and all other elements exactly as they are. Keep the same white background, table, lighting, shadows, and composition.',
    'Add a single yellow lemon in front of the red apple, leaving all existing fruits visible. Preserve the apple, banana, orange, pear, and everything else exactly as they are. Keep the white background, table, lighting, shadows, and camera angle unchanged.',
    'Add a single peach on the front-left area of the table, near the banana but not overlapping it. Preserve all existing fruits and all other elements exactly as they are, including the table, background, lighting, shadows, and composition.',
    'Add a single dark purple plum on the front-right area of the table, near the orange but not overlapping it. Preserve all existing fruits and the rest of the image exactly as they are.',
    'Add a small bunch of green grapes on the back-right area of the table, positioned so the grapes are fully visible. Preserve all previously existing fruits and keep the white background, table, lighting, shadows, and camera angle unchanged.',
    'Add a single large strawberry on the table near the lemon in the front-center area. Preserve all existing fruits and the rest of the image exactly as they are.',
    'Add a whole brown kiwi on the table near the peach on the left side. Preserve all other fruits and all other image elements exactly as they are, including lighting, shadows, background, and composition.',
    'Add a single mango on the back-left area of the table, balancing the composition while keeping all fruits visible. Preserve every previously added fruit and everything else in the image exactly as they are. Keep the same white background, table, lighting, shadows, and camera angle.',
]

In [3]:
def save_output(turn, interaction):
    assert interaction.output_image, f"turn {turn} returned no image"
    (API_DIR / f"{turn}.jpeg").write_bytes(base64.b64decode(interaction.output_image.data))
    print(f"turn {turn:2d} done  ({interaction.id})")


# Turn 0 is the image generated in the Gemini app, shared by both chains.
shutil.copy(APP_DIR / "0.jpeg", API_DIR / "0.jpeg")
start_image = base64.b64encode((APP_DIR / "0.jpeg").read_bytes()).decode()

# Turn 1 sends the starting image; every later turn chains on the previous
# interaction, so the model keeps the full editing history as context.
interaction = client.interactions.create(
    model=MODEL,
    input=[
        {"type": "text", "text": EDITS[0]},
        {"type": "image", "data": start_image, "mime_type": "image/jpeg"},
    ],
    response_format=RESPONSE_FORMAT,
    generation_config=GENERATION_CONFIG,
)
save_output(1, interaction)

for turn, prompt in enumerate(EDITS[1:], start=2):
    interaction = client.interactions.create(
        model=MODEL,
        input=prompt,
        previous_interaction_id=interaction.id,
        response_format=RESPONSE_FORMAT,
        generation_config=GENERATION_CONFIG,
    )
    save_output(turn, interaction)

turn  1 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXUk9XRmFzS3dOSUxfbnNFUGhvT0hzUW8)


turn  2 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXWC1XRmFydnRLWWJobnNFUDRQcWIwUVk)


turn  3 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXZXVXRmFzcmtCWWFxbnNFUG1zcTJ5QWM)


turn  4 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXbXVXRmFxR1NBb2JlN004UDVyZXp3UXM)


turn  5 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXdU9XRmF1UElMdG15bnNFUDhjTENvUWs)


turn  6 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXMC1XRmFybjdCX3VJbnNFUDVPQzg4QXM)


turn  7 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXOU9XRmFvaUFFSjM2bnNFUDQ5eWEtQXM)


turn  8 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXRmVhRmFxcmxEX3Z3bnNFUHdjTFI0QTA)


turn  9 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXTXVhRmFvN1BJSmk5bnNFUHVjM2RpQW8)


turn 10 done  (v1_ChdST1dGYXNLd05JTF9uc0VQaG9PSHNRbxIXVk9hRmFxVG1PYjNCbnNFUHFhTFd5UWs)


## Comparison

Pick the turn with the top slider, then drag the divider on the image:
left of the line is the **Gemini app** result, right is the **API** result.

In [3]:
from nanobanana.compare import comparison

comparison(APP_DIR, API_DIR, EDITS)